# Meridian Commerce Co. — Data Cleaning Pipeline
### `notebooks/01_data_cleaning.ipynb`

**Project:** Retail & E-commerce Analytics  
**Company (fictional):** Meridian Commerce Co. — omnichannel fashion & lifestyle retailer  
**Data range:** January 2023 – December 2024  
**Tables:** 5 raw CSVs produced by `scripts/generate_retail_data.py`

---

## What This Notebook Does

Raw data from operational systems (WMS, ERP, marketplace APIs) is almost never clean.
Before we can run any analysis, we need to:

1. **Understand** the structure and content of each table
2. **Detect** every data quality issue — without assuming we know what's wrong ahead of time
3. **Fix** each issue with a documented, reproducible decision
4. **Validate** our fixes didn't break referential integrity
5. **Save** a clean, analysis-ready version of each table

This notebook covers **all five steps** for all five tables in the retail dataset.

> 💡 **Why not just fix everything immediately?**  
> In real projects, you never know *all* the issues until you've looked at the data carefully.  
> Jumping straight to fixes risks missing problems you didn't think to check for.  
> We follow a structured checklist: **duplicate rows → missing values → value inconsistencies  
> → type issues → cross-table integrity → statistical sanity checks**.


## Dataset Overview

We're working with five relational tables. Each represents a different "slice" of how a
retail business generates and tracks data:

| Table | Source system | Key question it answers |
|---|---|---|
| `raw_products.csv` | Product catalog / ERP | What do we sell? |
| `raw_transactions.csv` | POS / marketplace API | What did we actually sell, when, where, and at what price? |
| `raw_inventory_snapshots.csv` | Warehouse Management System (WMS) | What's in stock? What's selling? What's sitting? |
| `raw_traffic_sessions.csv` | Google Analytics / Platform dashboard | Who's visiting our online stores? Are they buying? |
| `raw_marketing_spend.csv` | Ad platforms (Shopee Ads, Google, etc.) | What did we spend on marketing, and did it work? |

---

### Retail Terminology Reference

Since this is a retail dataset, we'll encounter industry-specific terms. Here are the key ones
defined once upfront — we'll revisit them in more detail when they appear:

| Term | Short definition |
|---|---|
| **SKU** | Stock Keeping Unit — the smallest trackable product unit (e.g., "Blue Shirt Size M" = 1 SKU) |
| **Channel** | Where a sale happened: Shopee, Tokopedia, Website (D2C), or Offline (physical store) |
| **STR** | Sell-Through Rate — % of available inventory that was sold in a period |
| **DOI** | Days of Inventory — at current sales pace, how many days until stock runs out |
| **Lead Time** | Days from placing a restock order until the goods arrive |
| **Stockout** | When a SKU's inventory hits zero while demand still exists (= lost sales) |
| **COGS** | Cost of Goods Sold — what it cost Meridian to make/buy the items that were sold |
| **Gross Profit** | Revenue − COGS (profit before platform fees, shipping, operating costs) |
| **ROAS** | Return on Ad Spend — revenue generated per rupiah spent on advertising |
| **CTR** | Click-Through Rate — % of people who saw an ad and clicked on it |
| **CVR / Conversion Rate** | % of website visitors who complete a purchase |
| **AOV** | Average Order Value — mean transaction value |


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

# Suppress minor pandas warnings to keep output clean
warnings.filterwarnings("ignore")

# Display settings — show more columns and rows by default
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries loaded. Pandas version:", pd.__version__)


## Step 0 — Load the Raw Data

We load all five tables as strings first (`dtype=str`). This is a deliberate choice:

> **Why load as strings?**  
> If we let pandas infer data types at load time, it might silently coerce messy values
> (e.g., it could turn a malformed date into NaT, or a text price like "16.75" into a float
> before we even know it should be an integer). By loading everything as strings, we see
> exactly what's in the file before any transformation happens.


In [ ]:
# -- Load all five raw tables as raw strings --------------------------------

RAW_DIR = "../data/raw"

products   = pd.read_csv(os.path.join(RAW_DIR, "raw_products.csv"),            dtype=str)
txn        = pd.read_csv(os.path.join(RAW_DIR, "raw_transactions.csv"),        dtype=str)
inventory  = pd.read_csv(os.path.join(RAW_DIR, "raw_inventory_snapshots.csv"), dtype=str)
traffic    = pd.read_csv(os.path.join(RAW_DIR, "raw_traffic_sessions.csv"),    dtype=str)
marketing  = pd.read_csv(os.path.join(RAW_DIR, "raw_marketing_spend.csv"),     dtype=str)

# Quick size overview — how many rows and columns do we have?
tables = {
    "products":            products,
    "transactions":        txn,
    "inventory_snapshots": inventory,
    "traffic_sessions":    traffic,
    "marketing_spend":     marketing,
}

print(f"{'Table':<25} {'Rows':>8} {'Columns':>10}")
print("-" * 45)
for name, df in tables.items():
    print(f"{name:<25} {len(df):>8,} {df.shape[1]:>10}")

total = sum(len(df) for df in tables.values())
print("-" * 45)
print(f"{'TOTAL':<25} {total:>8,}")


---
## Section 1 — Initial Exploration

Before touching anything, we do a complete first pass across all five tables.
The goal here is **understanding**, not fixing.

We want to know:
- What columns exist?
- What do the first few rows look like?
- Are the column names self-explanatory?
- Are there any obviously weird values visible in the head?

Think of this as reading the manual before operating a machine.


### 1.1 Products Table

This is the master catalog of everything Meridian sells.
Each row = one SKU (Stock Keeping Unit).

> 📌 **Retail term — SKU**  
> Imagine a T-shirt that comes in 3 colors and 4 sizes. That's 12 SKUs, not 1 product.  
> Each SKU has its own cost, price, and inventory level. Most retail systems track  
> everything at the SKU level, not the product level.


In [ ]:
# Look at the first few rows of the products table
# .head(5) shows the first 5 rows — a quick "what does this look like?" check
products.head(5)


In [ ]:
# Check column names and data types
# Since we loaded as dtype=str, everything will show as 'object' — that's expected
print("Columns and types:")
print(products.dtypes)
print(f"\nShape: {products.shape[0]} rows × {products.shape[1]} columns")


### 1.2 Transactions Table

The most important table in the dataset. Every row = one product sold (line item).

> 📌 **Retail term — Channel**  
> Where the sale happened. Meridian operates on four channels:  
> - **Shopee / Tokopedia** — marketplace platforms. Meridian pays a platform fee (~3-6% of GMV).  
> - **Website** — Meridian's own D2C (Direct-to-Consumer) store. Lower fees, higher margin.  
> - **Offline** — Physical stores. No platform fee, but higher fixed costs (rent, staff).  
>   
> Channel strategy (which channel to invest in) is one of the most important decisions  
> in retail. The data here will help us quantify the margin difference.

> 📌 **Retail term — Gross Profit vs Net Profit**  
> `gross_profit_idr = revenue_idr − cogs_idr`  
> This is profit BEFORE deducting platform fees, shipping, marketing, and operating costs.  
> Gross Profit Margin (GP%) = gross_profit / revenue.  
> A typical fashion retailer targets GP% of 50-65%. Below 40% is a red flag.


In [ ]:
# Look at the transactions table
txn.head(5)


In [ ]:
# How many unique values in key categorical columns?
# This gives us a sense of the data's scope
print("Unique customers:", txn["customer_id"].nunique())
print("Unique SKUs sold:", txn["sku_id"].nunique())
print("Unique channels (raw):", txn["channel"].nunique(), "—", txn["channel"].unique().tolist())
print("Unique provinces (raw):", txn["province"].nunique())
print("Date range (raw):", txn["transaction_date"].min(), "to", txn["transaction_date"].max())


### 1.3 Inventory Snapshots Table

Monthly snapshot of stock levels for each active SKU.
Each row = one SKU in one month.

> 📌 **Retail term — Sell-Through Rate (STR)**  
> `STR = sold_qty / (opening_stock + incoming_qty)`  
> This answers: "Of all the inventory we had available this month, what % did we actually sell?"  
>   
> Benchmarks for fashion retail:  
> - **STR > 60%**: Healthy. Product is moving well.  
> - **STR 30-60%**: OK. Watch and reorder carefully.  
> - **STR < 20%**: Warning. This is slow-moving inventory.  
> - **STR < 10%**: Deadstock risk. Capital is locked in unselling goods.  
>   
> If STR is consistently low AND you keep restocking, you're literally buying your own  
> overstock problem. This is one of the most common (and expensive) mistakes in retail.

> 📌 **Retail term — Days of Inventory (DOI)**  
> `DOI = closing_stock / average_daily_sales`  
> "How many more days can I keep selling at this pace before I run out?"  
>   
> DOI 30 = You'll run out in a month → might need to reorder soon.  
> DOI 200 = You have 6+ months of stock → probably too much.  
> DOI 999 = Essentially infinite → something is very wrong (deadstock).

> 📌 **Retail term — Stockout (is_stockout = 1)**  
> When `closing_stock = 0` AND `sold_qty > 0`, it means you ran out of stock  
> while customers were still trying to buy. This = **lost revenue**.  
> Every stockout event is money you couldn't collect because you didn't have the product.


In [ ]:
# Quick look at inventory table
inventory.head(5)


In [ ]:
# Check the range of key metrics (still as strings — just checking values)
print("Months covered:", inventory["year_month"].nunique(), "unique months")
print("SKUs tracked:", inventory["sku_id"].nunique())
print("Unique subcategories:", inventory["subcategory"].unique().tolist())
print("\nStockout events (raw count):", (inventory["is_stockout"] == "1").sum())


### 1.4 Traffic Sessions Table

Daily session data per online channel. This table is what connects
"how many people visited?" to "how many actually bought?"

> 📌 **Retail term — Conversion Rate (CVR)**  
> `CVR = orders / sessions`  
> The single most important e-commerce efficiency metric.  
>   
> Example: 1,000 people visited your Shopee store today. 120 bought something.  
> CVR = 120 / 1,000 = **12%**.  
>   
> Industry benchmarks for fashion e-commerce:  
> - 1-3%: Typical for most online stores  
> - 5-10%: Very good  
> - >15%: Usually driven by a promotion event  
>   
> A sudden CVR spike is often a promo. A CVR drop could mean poor product-market fit,  
> price issue, or competitive pressure.

> 📌 **Retail term — Average Order Value (AOV)**  
> `AOV = total_revenue / number_of_orders`  
> How much does the average customer spend per visit?  
> Higher AOV generally means better unit economics (you're spending marketing to  
> acquire a customer who buys more). Discount promos often increase orders but  
> decrease AOV because people spend less per transaction.


In [ ]:
# Traffic sessions overview
traffic.head(5)


In [ ]:
# Check channels and promo flags
print("Online channels:", traffic["channel"].unique().tolist())
print("Date range:", traffic["session_date"].min(), "to", traffic["session_date"].max())
print("Is_promo_period distribution:\n", traffic["is_promo_period"].value_counts().to_string())


### 1.5 Marketing Spend Table

Weekly marketing spend per advertising platform. This table answers:
"Is our advertising actually working?"

> 📌 **Retail term — ROAS (Return on Ad Spend)**  
> `ROAS = revenue_attributed / spend`  
>   
> If you spent Rp 5,000,000 on Shopee Ads and those ads are credited with generating  
> Rp 17,500,000 in revenue: ROAS = 17.5M / 5M = **3.5x**.  
>   
> Interpretation: For every Rp1 you spent on ads, you got Rp3.5 back in revenue.  
>   
> A ROAS of 1.0 = break-even on ad spend alone (you still have COGS, fees, etc. to cover).  
> Most retailers need ROAS > 3x to be profitable after all costs.  
>   
> ⚠️ **Important caveat**: Attribution is imperfect. "Revenue attributed" means the last  
> ad click before a purchase gets credit — but the customer may have seen many ads  
> before that final click. ROAS should be read as directional, not exact.

> 📌 **Retail term — CTR (Click-Through Rate)**  
> `CTR = clicks / impressions`  
> Of all the people who *saw* your ad, what % actually *clicked* on it?  
> Low CTR = your ad creative isn't compelling. High CTR = people want to see more.


In [ ]:
# Marketing spend overview
marketing.head(5)


In [ ]:
# Platform breakdown
print("Ad platforms:", marketing["platform"].unique().tolist())
print("Weekly date range:", marketing["week_start"].min(), "to", marketing["week_start"].max())
print("Total campaigns/rows:", len(marketing))


---
## Section 2 — Systematic Cleaning Checklist

Now that we understand the data, we work through a standardized checklist.
We'll apply each check **to all five tables** before moving on — not just to the
tables we *think* might have a problem.

| # | Check | Tables |
|---|---|---|
| 2.1 | Duplicate rows | All 5 |
| 2.2 | Missing (null) values | All 5 |
| 2.3 | Channel name inconsistencies | Transactions, Traffic |
| 2.4 | Price anomalies (statistical outlier detection) | Transactions |
| 2.5 | Negative quantities | Transactions |
| 2.6 | Date format inconsistencies | Transactions, Traffic, Inventory |
| 2.7 | Province name inconsistencies | Transactions |
| 2.8 | Product name formatting | Products |
| 2.9 | Cross-table referential integrity | All 5 |
| 2.10 | Statistical sanity checks | Transactions, Inventory, Traffic |

---

### 2.1 — Duplicate Row Detection

**Hypothesis:** We don't know yet which tables have duplicates. We'll check all five.

Why do duplicates happen in real data?
- Database sync errors (a record gets written twice during a system restart)
- ETL pipeline bugs (data extraction runs twice)
- Manual data entry (someone enters the same row in a spreadsheet)
- API pagination errors (the same page of results is fetched twice)

> **Our approach:** Check `duplicated()` across all tables. If duplicates exist,
> we investigate *what kind* they are before dropping — sometimes what looks like
> a duplicate is actually a legitimate repeated event.


In [ ]:
# -- Check for duplicate rows in ALL five tables ---------------------------
# We check ALL tables systematically. We don't skip any, even if we "expect"
# some to be clean — confirming cleanliness is also valuable information.

print("=" * 60)
print("DUPLICATE ROW CHECK — ALL TABLES")
print("=" * 60)

for name, df in tables.items():
    n_total    = len(df)
    n_dupes    = df.duplicated().sum()
    dupe_pct   = n_dupes / n_total * 100 if n_total > 0 else 0
    status     = "⚠️  HAS DUPLICATES" if n_dupes > 0 else "✓  clean"
    print(f"  {name:<25} {n_dupes:>6} / {n_total:>7} duplicates ({dupe_pct:.2f}%)  {status}")


We found that `transactions` has duplicate rows. Let's investigate before we remove them.

**Rule:** Never drop data without first understanding *why* it's there.

Is it an exact duplicate (same transaction ID, same everything)?
Or does the same transaction ID have different values in other columns?
These are two very different problems with different fixes.


In [ ]:
# -- Investigate the duplicate transactions ---------------------------------

# Step 1: Find the duplicated rows themselves
dupe_mask   = txn.duplicated(keep=False)   # keep=False marks ALL copies, not just second
dupe_rows   = txn[dupe_mask].copy()

print(f"Total rows involved in duplication: {len(dupe_rows)}")
print(f"That represents {dupe_rows['transaction_id'].nunique()} unique transaction IDs\n")

# Step 2: Are they EXACT duplicates (every column identical)?
# We compare keeping='first' vs full duplication
exact_dupes = txn.duplicated(keep="first").sum()
print(f"Exact row duplicates (every column identical): {exact_dupes}")
print()

# Step 3: Look at a sample of the duplicated rows to understand the pattern
print("Sample duplicated rows (showing 4 examples — 2 pairs):")
sample_ids = dupe_rows["transaction_id"].iloc[:4].tolist()
txn[txn["transaction_id"].isin(sample_ids)].sort_values("transaction_id").head(8)


In [ ]:
# These are exact duplicates — the full row is repeated.
# Decision: DROP the second copy, keep the first occurrence.
#
# Why first? In database terms, if we have two identical rows,
# we have no way to know which is "real." We arbitrarily keep one.
# The important thing is that we DOCUMENT this decision.
#
# Alternative approaches:
#   - Keep 'last' (same outcome for exact duplicates, different for partial)
#   - Flag duplicates and quarantine for manual review
#   - Keep 'first' and log the count (our choice)

n_before = len(txn)
txn      = txn.drop_duplicates(keep="first").reset_index(drop=True)
n_after  = len(txn)

print(f"Rows before: {n_before:,}")
print(f"Rows after:  {n_after:,}")
print(f"Removed:     {n_before - n_after:,} duplicate rows")


In [ ]:
# -- Verify: confirm zero duplicates remain ---------------------------------
remaining_dupes = txn.duplicated().sum()
print(f"Remaining duplicates in transactions: {remaining_dupes}")
print("✓  Duplicate check passed." if remaining_dupes == 0 else "⚠️  Still has duplicates!")


---
### 2.2 — Missing Value Analysis

Missing values come in two flavors in real datasets:
1. **Expected missing** — columns that are intentionally empty for certain rows  
   (e.g., `store_id` is NULL for online transactions — there's no physical store)
2. **Unexpected missing** — columns that SHOULD have a value but don't  
   (e.g., `store_id` is NULL for an *offline* transaction — that's a problem)

This distinction is crucial. Blindly filling all NULLs or dropping all rows with NULLs
destroys valid data. We need to understand WHY values are missing before acting.


In [ ]:
# -- Missing value count across ALL tables ----------------------------------
print("=" * 70)
print("MISSING VALUE COUNT — ALL TABLES")
print("=" * 70)

for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]  # only show columns that actually have missing values

    if len(missing) == 0:
        print(f"\n{name}: ✓  No missing values")
    else:
        print(f"\n{name}: ⚠️  {missing.sum()} total missing values")
        for col, count in missing.items():
            pct = count / len(df) * 100
            print(f"    {col:<30} {count:>6,} missing ({pct:.1f}%)")


Let's look at the `transactions` table more carefully —
it has the most complex missing value pattern.

We have missing values in `store_id`, `city`, and `promo_code`.
Let's investigate which are *expected* and which are *problems*.


In [ ]:
# -- Analyze missing values in transactions by context ---------------------

# store_id and city are for offline transactions only.
# Online transactions shouldn't have a store_id — that's EXPECTED missing.
# Let's verify: do all missing store_ids belong to online channels?

# First: standardize channel values temporarily just for this analysis
# (we'll do the full clean in step 2.3)
txn_check = txn.copy()
txn_check["channel_lower"] = txn_check["channel"].str.lower().str.strip()

# Online channels
online_mask  = txn_check["channel_lower"].isin(["shopee", "shopee.co.id", "tokopedia",
                                                   "toped", "website", "web"])
offline_mask = txn_check["channel_lower"].isin(["offline", "in-store", "offline"])

store_null   = txn_check["store_id"].isnull()

print("Missing store_id analysis:")
print(f"  Online transactions with NULL store_id:  {(online_mask & store_null).sum():,}  ← EXPECTED")
print(f"  Offline transactions with NULL store_id: {(offline_mask & store_null).sum():,}  ← PROBLEM")
print()

# promo_code: should be NULL when is_promo = False
promo_null = txn_check["promo_code"].isnull()
is_promo_f = txn_check["is_promo"] == "False"
is_promo_t = txn_check["is_promo"] == "True"

print("Missing promo_code analysis:")
print(f"  Non-promo transactions (is_promo=False) with NULL promo_code: {(is_promo_f & promo_null).sum():,}  ← EXPECTED")
print(f"  Promo transactions (is_promo=True) with NULL promo_code:      {(is_promo_t & promo_null).sum():,}  ← could be partial promo")


In [ ]:
# -- Decision for each missing value type ----------------------------------
#
# store_id for online = KEEP AS NULL (it's correct — no store for online orders)
# store_id for offline = UNKNOWN — we can't recover this. Flag it.
# city for online      = KEEP AS NULL (online orders have province but not city)
# promo_code for non-promo = KEEP AS NULL (no promo code = no code)
# promo_code for promo = These may be transactions with unnamed/system promos.
#                        We'll keep them as-is and document.

# Flag the problematic offline rows with missing store_id
offline_no_store = offline_mask & txn_check["store_id"].isnull()
txn.loc[offline_no_store.values, "data_quality_flag"] = "missing_store_id"

flagged_count = (txn.get("data_quality_flag", pd.Series()) == "missing_store_id").sum()
print(f"Flagged {flagged_count} offline rows with missing store_id for downstream review")
print("These rows are kept in the clean dataset but flagged — they're still useful for")
print("sales analysis even without a specific store identifier.")


---
### 2.3 — Channel Name Standardization

Channel is a core dimension in retail analytics. Virtually every analysis
will GROUP BY channel to compare Shopee vs Tokopedia vs Offline.

If channel names are inconsistent ("shopee" vs "SHOPEE" vs "Shopee.co.id"),
grouping will split one channel into multiple fake groups — your Shopee revenue will
appear fragmented across 3+ rows instead of summing into one.

Let's check what we have.


In [ ]:
# -- Examine all unique channel values across both tables that use 'channel' --

for table_name in ["transactions", "traffic_sessions"]:
    df = tables[table_name]
    if "channel" in df.columns:
        uniq = df["channel"].value_counts()
        print(f"\n--- {table_name} ---")
        print(f"  Found {len(uniq)} unique channel values:")
        for val, cnt in uniq.items():
            print(f"    '{val}' → {cnt:,} rows")


We can see the problem clearly: `Shopee`, `shopee`, `SHOPEE`, and `Shopee.co.id`
are all the same channel — but they'll be treated as 4 separate channels in any GROUP BY.

We build a **mapping dictionary**: every variant → its canonical (standardized) name.

> **Tip:** Always map TO a simple, consistent name.
> `"Shopee"` not `"shopee"` (capitalize for readability in charts).
> `"Offline"` not `"In-Store"` (use the shorter name unless there's a business reason otherwise).


In [ ]:
# -- Build channel mapping and apply ----------------------------------------

CHANNEL_MAP = {
    # Shopee variants
    "shopee":        "Shopee",
    "SHOPEE":        "Shopee",
    "Shopee.co.id":  "Shopee",
    # Tokopedia variants
    "tokopedia":     "Tokopedia",
    "TOKOPEDIA":     "Tokopedia",
    "Toped":         "Tokopedia",
    # Website variants
    "website":       "Website",
    "WEBSITE":       "Website",
    "Web":           "Website",
    # Offline variants
    "offline":       "Offline",
    "OFFLINE":       "Offline",
    "In-Store":      "Offline",
    # Already-clean values (map to themselves so they don't become NaN)
    "Shopee":        "Shopee",
    "Tokopedia":     "Tokopedia",
    "Website":       "Website",
    "Offline":       "Offline",
}

# Apply to transactions
txn["channel"] = txn["channel"].map(CHANNEL_MAP)

# Apply to traffic_sessions (also has channel column)
traffic["channel"] = traffic["channel"].map(CHANNEL_MAP)

# -- Verify ------------------------------------------------------------------
print("Transactions — unique channels after cleaning:")
print(txn["channel"].value_counts().to_string())
print()
print("Traffic — unique channels after cleaning:")
print(traffic["channel"].value_counts().to_string())


In [ ]:
# -- Check: did any values get lost (mapped to NaN)? --------------------------
null_channels_txn = txn["channel"].isnull().sum()
null_channels_trf = traffic["channel"].isnull().sum()

print(f"NULL channels after mapping — transactions: {null_channels_txn}")
print(f"NULL channels after mapping — traffic:      {null_channels_trf}")

if null_channels_txn > 0 or null_channels_trf > 0:
    print("\n⚠️  Some values weren't mapped. Investigate:")
    unmapped_txn = txn[txn["channel"].isnull()]["channel"].unique()
    print("  Unmapped in transactions:", unmapped_txn)
else:
    print("\n✓  All channel values mapped successfully. No data lost.")


---
### 2.4 — Price Anomaly Detection (Statistical Approach)

Here's a scenario that happens in real retail systems: someone exports data from an
accounting system configured for USD, pastes it into an IDR-denominated spreadsheet,
and the numbers look fine on screen (they're valid floats) but they're wrong by a
factor of ~15,000.

We don't know ahead of time *which* rows have this problem, or how many.
We can't look at 58,000 rows manually. We need a statistical approach to find them.

**Method: IQR-based outlier detection**

> 📌 **Statistical concept — IQR (Interquartile Range)**  
> Sort all values from smallest to largest.  
> - Q1 = the value at the 25th percentile (bottom quarter)  
> - Q3 = the value at the 75th percentile (top quarter)  
> - IQR = Q3 − Q1 (the "middle 50% spread")  
>   
> The standard outlier rule: anything below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`  
> is flagged as a statistical outlier. These aren't necessarily errors — but they are  
> unusual enough to investigate.  
>   
> For our price problem: USD-converted prices will be around 10-20 (IDR equivalent ~150k-300k).  
> Normal IDR prices are 50,000 to 800,000+. The statistical method should clearly  
> separate these two populations.


In [ ]:
# -- Step 1: Convert price columns to numeric and inspect distributions --------
# We need to convert to numbers first to do statistical analysis.
# errors='coerce' = if a value can't be converted to a number, make it NaN
# (we'll catch those NaNs separately)

price_cols = ["base_price_idr", "final_price_idr", "revenue_idr",
              "cogs_idr", "gross_profit_idr"]

# Temporarily convert to float just for exploration (we'll handle types properly in step 2.9)
for col in price_cols:
    txn[col] = pd.to_numeric(txn[col], errors="coerce")

print("Distribution of base_price_idr (in IDR):")
print(txn["base_price_idr"].describe().apply(lambda x: f"{x:,.2f}").to_string())


In [ ]:
# -- Step 2: Use IQR to identify statistical outliers --------------------------

col   = "base_price_idr"
q1    = txn[col].quantile(0.25)
q3    = txn[col].quantile(0.75)
iqr   = q3 - q1

# Standard fence multiplier is 1.5 for "mild" outliers, 3.0 for "extreme"
# For currency errors (USD vs IDR), a 1.5x fence is more than enough —
# the gap between 16.75 (USD) and 75,000 (IDR) is enormous.
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1 (25th percentile): {q1:,.0f} IDR")
print(f"Q3 (75th percentile): {q3:,.0f} IDR")
print(f"IQR:                  {iqr:,.0f} IDR")
print(f"Lower fence (Q1 - 1.5×IQR): {lower_fence:,.2f} IDR")
print(f"Upper fence (Q3 + 1.5×IQR): {upper_fence:,.0f} IDR")
print()

# How many rows fall BELOW the lower fence?
below_lower = txn[txn[col] < lower_fence]
print(f"Rows below lower fence (< {lower_fence:.2f}): {len(below_lower)}")
if len(below_lower) > 0:
    print("\nSample of these suspicious rows:")
    print(below_lower[[col, "final_price_idr", "revenue_idr", "channel", "sku_id"]].head(10).to_string())


In [ ]:
# -- Step 3: Confirm the USD hypothesis ----------------------------------------
# Values around 10-20 with a 1 USD ≈ 15,000 IDR conversion rate → original IDR was ~150k-300k
# That's plausible for fashion items, confirming the USD problem

usd_threshold = 1_000  # Any price_idr < 1,000 is almost certainly wrong for a fashion item
usd_mask      = txn["base_price_idr"] < usd_threshold

print(f"Rows with base_price_idr < {usd_threshold:,}: {usd_mask.sum()}")
print()
print("Value distribution of these suspicious rows:")
print(txn.loc[usd_mask, "base_price_idr"].describe().apply(lambda x: f"{x:.4f}").to_string())
print()
print("Cross-check with other price columns (should all be proportionally small):")
print(txn.loc[usd_mask, price_cols].describe().apply(lambda x: f"{x:.4f}").to_string())


In [ ]:
# -- Step 4: Decision — how to handle USD rows ---------------------------------
#
# Options:
#   A) Convert back to IDR: multiply by ~15,000
#      Problem: we don't know the exact exchange rate on each transaction date.
#               Getting it wrong could be worse than removing the rows.
#   B) Drop these rows entirely
#      Problem: we lose data (but only ~0.05% of rows)
#   C) Flag and keep as-is, exclude from price-sensitive analyses
#      Problem: easy to accidentally include them in a sum/average
#
# Decision: OPTION B — DROP.
# Reason: 30 rows out of 58,000+ = 0.05% of data.
#         The revenue error if included is orders of magnitude larger than the
#         statistical loss from dropping 30 rows. When in doubt, discard bad data
#         rather than carry a known error forward.

n_before = len(txn)
txn      = txn[~usd_mask].reset_index(drop=True)
n_after  = len(txn)

print(f"Rows before removing USD anomalies: {n_before:,}")
print(f"Rows after:                         {n_after:,}")
print(f"Removed:                            {n_before - n_after:,} rows ({(n_before-n_after)/n_before*100:.3f}%)")


In [ ]:
# -- Verify: check the distribution looks reasonable now ----------------------
print("Distribution of base_price_idr after removing USD anomalies:")
print(txn["base_price_idr"].describe().apply(lambda x: f"{x:,.0f}").to_string())
print()
print("Minimum value (should be a sensible IDR amount):", f"{txn['base_price_idr'].min():,.0f}")


---
### 2.5 — Negative Quantity Investigation

In retail, a negative quantity in a sales transaction usually means one of:
1. A **product return** was recorded as a negative sale instead of a separate return record
2. A **data entry error** (someone typed −2 instead of 2)
3. An **inventory adjustment** that got mixed into the sales table

All three are problematic in a transaction table. Let's investigate which applies here.


In [ ]:
# -- Check for negative quantities --------------------------------------------

# Convert quantity to numeric first
txn["quantity"] = pd.to_numeric(txn["quantity"], errors="coerce")

neg_mask   = txn["quantity"] < 0
n_negative = neg_mask.sum()

print(f"Rows with negative quantity: {n_negative}")
print()

if n_negative > 0:
    neg_rows = txn[neg_mask]
    print("Distribution of negative quantities:")
    print(neg_rows["quantity"].value_counts().sort_index().to_string())
    print()
    print("Do they cluster in specific channels or subcategories?")
    print("\nBy channel:")
    print(neg_rows["channel"].value_counts().to_string())
    print("\nBy subcategory:")
    print(neg_rows["subcategory"].value_counts().to_string())
    print()
    print("Sample rows:")
    print(neg_rows[["transaction_id","transaction_date","channel",
                     "subcategory","quantity","revenue_idr","is_promo"]].head(8).to_string())


In [ ]:
# -- Decision: What do we do with negative quantity rows? ---------------------
#
# Findings:
# - They're spread across channels and subcategories (no pattern = not a systematic issue)
# - The revenue_idr is also negative for these rows (quantity × price = negative revenue)
# - This strongly suggests mis-recorded returns, NOT intentional negatives
#
# Options:
#   A) Take absolute value (abs): treats them as regular sales
#      Problem: if these ARE returns, we're double-counting the revenue
#   B) Drop them: we lose 25 rows but maintain clean sales data
#      This is appropriate if we're building a sales analysis (not a returns analysis)
#   C) Move to a separate returns table
#      Best practice for production, but out of scope for this project
#
# Decision: OPTION B — DROP.
# Reason: This notebook builds a sales transaction table. Returns analysis would
# require a separate returns table with different logic. We document the count (25)
# so a future analyst can retrieve these from the raw file if needed.

n_before = len(txn)
txn      = txn[txn["quantity"] > 0].reset_index(drop=True)
n_after  = len(txn)

print(f"Removed {n_before - n_after} negative-quantity rows.")
print(f"Transactions remaining: {n_after:,}")


---
### 2.6 — Date Format Standardization

Date columns are deceptively tricky. The value `"05/06/2024"` could mean:
- May 6th (MM/DD/YYYY format — US convention)
- June 5th (DD/MM/YYYY format — Indonesian/European convention)

Without knowing which format convention was used, **this date is ambiguous and dangerous**.
Fortunately, if a value has a day > 12 (e.g., `"25/06/2024"`), we know it *must* be
DD/MM/YYYY — because no month goes above 12.

We'll apply a `parse_date()` function that tries multiple formats in sequence,
using the most unambiguous format (ISO: YYYY-MM-DD) first.


In [ ]:
# -- Step 1: Try pandas standard date parsing and see what fails --------------

# pd.to_datetime with errors='coerce' returns NaT (Not a Time) for values it can't parse.
# We first try the ISO format (YYYY-MM-DD), which is unambiguous.
test_parse = pd.to_datetime(txn["transaction_date"], format="%Y-%m-%d", errors="coerce")

failed_iso = txn.loc[test_parse.isna(), "transaction_date"]
print(f"Rows that failed ISO (YYYY-MM-DD) parsing: {test_parse.isna().sum()}")
print()
if len(failed_iso) > 0:
    print("Sample non-ISO date values:")
    # Show unique examples of the formats we need to handle
    print(failed_iso.drop_duplicates().head(15).tolist())


In [ ]:
# -- Step 2: Identify the alternative formats present --------------------------

# From the sample, we can see two non-ISO formats:
# Format A: "DD/MM/YYYY"  e.g., "25/06/2024"
# Format B: "Month DD, YYYY"  e.g., "June 25, 2024"
#
# There might be others. The safest approach is a function that TRIES each format
# in sequence and returns the first successful parse. The ISO format goes FIRST
# because it's unambiguous.

from datetime import date as pydate

def parse_date(value: str) -> str:
    # Attempt to parse a date string using multiple format patterns.
    # Returns ISO format (YYYY-MM-DD) on success, or np.nan on failure.
    #
    # Why multiple formats? Real data comes from multiple source systems
    # with different locale settings. We try ISO first (unambiguous), then
    # European (DD/MM), then long-form English (June 25, 2024).
    if pd.isna(value) or str(value).strip() == "":
        return np.nan

    value = str(value).strip()

    # Try each known format in order of reliability / unambiguity
    formats = [
        "%Y-%m-%d",     # ISO standard: 2024-06-25 (most common, most reliable)
        "%d/%m/%Y",     # European/Indonesian: 25/06/2024
        "%m/%d/%Y",     # US format: 06/25/2024 (only safe when day > 12)
        "%B %d, %Y",    # English long: June 25, 2024
        "%b %d, %Y",    # English short: Jun 25, 2024
        "%d-%m-%Y",     # Dashed European: 25-06-2024
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
        except (ValueError, TypeError):
            continue

    # If none of the formats work, return NaN — we'll catch these separately
    return np.nan


# Test on a few examples
test_cases = ["2024-06-25", "25/06/2024", "June 25, 2024", "Jun 25, 2024", "nonsense"]
print("parse_date() test:")
for tc in test_cases:
    result = parse_date(tc)
    print(f"  '{tc}' → '{result}'")


In [ ]:
# -- Step 3: Apply parse_date to all date columns ------------------------------

# For efficiency, we use apply() which runs parse_date on each row.
# For production at scale, vectorized approaches are faster — but for a
# ~58k row dataset, apply() is fine and more readable.

txn["transaction_date"] = txn["transaction_date"].apply(parse_date)
inventory["year_month"] = inventory["year_month"].str[:7]  # already in YYYY-MM, just trim

# Parse traffic and marketing dates too
traffic["session_date"]    = traffic["session_date"].apply(parse_date)
marketing["week_start"]    = marketing["week_start"].apply(parse_date)
products["launch_date"]    = products["launch_date"].apply(parse_date)

# -- Verify: how many dates still couldn't be parsed? ------------------------
n_failed_txn = txn["transaction_date"].isna().sum()
n_failed_trf = traffic["session_date"].isna().sum()
n_failed_mkt = marketing["week_start"].isna().sum()

print(f"Dates that couldn't be parsed:")
print(f"  transactions:    {n_failed_txn}")
print(f"  traffic:         {n_failed_trf}")
print(f"  marketing_spend: {n_failed_mkt}")

if n_failed_txn + n_failed_trf + n_failed_mkt == 0:
    print("\n✓  All dates parsed successfully.")
else:
    print("\n⚠️  Some dates still couldn't be parsed. Sample unresolved values:")
    print(txn.loc[txn["transaction_date"].isna(), "transaction_date"].head())


In [ ]:
# -- Step 4: Validate date range makes sense -----------------------------------
# All transaction dates should be within Jan 2023 – Dec 2024 (our data window)

EXPECTED_START = "2023-01-01"
EXPECTED_END   = "2024-12-31"

out_of_range = txn[
    (txn["transaction_date"] < EXPECTED_START) |
    (txn["transaction_date"] > EXPECTED_END)
]

print(f"Transactions outside expected date range ({EXPECTED_START} to {EXPECTED_END}):")
print(f"  Count: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range["transaction_date"].value_counts().head())
else:
    print("  ✓  None. All dates are within the expected range.")


---
### 2.7 — Province Name Standardization

Geographic data is essential in retail analytics — it drives decisions about
store expansion, regional marketing spend, and warehouse placement.

But geographic names are notoriously inconsistent in real data:
- Systems get switched between Indonesian and English labels
- Operators use abbreviations (Jabar, Jatim)
- Data entry uses ALL CAPS in some systems

If we leave these inconsistent, any geographic analysis will show:
- "DKI Jakarta" with 30,000 sales
- "Jakarta" with 5,000 sales  
- "JAKARTA" with 3,000 sales

...when all three are the same province.


In [ ]:
# -- Step 1: Examine all unique province values --------------------------------
# We look at ALL province values and their frequency.
# High-frequency variants are the ones that will most distort our analysis.

all_provinces = txn["province"].value_counts()
print(f"Unique province values in transactions: {len(all_provinces)}")
print()
print(all_provinces.to_string())


In [ ]:
# -- Step 2: Build province mapping dictionary ---------------------------------
# Standard: use the official Indonesian province name (BPS standard)
# If it's already correct, map it to itself (so we don't accidentally drop it)

PROVINCE_MAP = {
    # DKI Jakarta variants
    "DKI Jakarta":    "DKI Jakarta",
    "Jakarta":        "DKI Jakarta",
    "JAKARTA":        "DKI Jakarta",
    "Dki Jakarta":    "DKI Jakarta",
    "Jakarta (DKI)":  "DKI Jakarta",
    # Jawa Barat variants
    "Jawa Barat":     "Jawa Barat",
    "Jabar":          "Jawa Barat",
    "JAWA BARAT":     "Jawa Barat",
    "West Java":      "Jawa Barat",
    "Jawa barat":     "Jawa Barat",
    # Jawa Timur variants
    "Jawa Timur":     "Jawa Timur",
    "Jatim":          "Jawa Timur",
    "JAWA TIMUR":     "Jawa Timur",
    "East Java":      "Jawa Timur",
    "jawa timur":     "Jawa Timur",
    # Jawa Tengah variants
    "Jawa Tengah":    "Jawa Tengah",
    "Jateng":         "Jawa Tengah",
    "JAWA TENGAH":    "Jawa Tengah",
    "Central Java":   "Jawa Tengah",
    "Jawa tengah":    "Jawa Tengah",
    # Banten variants
    "Banten":         "Banten",
    "BANTEN":         "Banten",
    "banten":         "Banten",
    # Others (clean — map to themselves)
    "Sumatera Utara":  "Sumatera Utara",
    "Sumatera Selatan":"Sumatera Selatan",
    "Riau":            "Riau",
    "Kepulauan Riau":  "Kepulauan Riau",
    "Kalimantan Timur":"Kalimantan Timur",
    "Sulawesi Selatan":"Sulawesi Selatan",
    "Bali":            "Bali",
    "DI Yogyakarta":   "DI Yogyakarta",
    "Lainnya":         "Lainnya",
    # Offline stores have province from the store table (should already be clean)
    # but include just in case
}

# Apply mapping
txn["province"] = txn["province"].map(PROVINCE_MAP)

# Check for unmapped values (would become NaN after map())
n_unmapped = txn["province"].isnull().sum()
print(f"Province values that couldn't be mapped (became NULL): {n_unmapped}")
if n_unmapped > 0:
    print("Unmapped values (add to PROVINCE_MAP if legitimate):")
    print(txn.loc[txn["province"].isnull(), "province"].unique())


In [ ]:
# -- Verify province distribution after cleaning -------------------------------
print("Province distribution after standardization:")
print(txn["province"].value_counts().to_string())


---
### 2.8 — Product Name Formatting (Products Table)

Product names aren't used for analytics calculations, but they matter for:
- Dashboard labels (ALLCAPS names look unprofessional in charts)
- Text search (finding "Slim-Fit Oxford Shirt" requires consistent casing)
- Data joining (if other systems reference product names, case must match)

The fix here is simple: `str.strip()` (remove leading/trailing whitespace)
followed by `str.title()` (capitalize first letter of each word).


In [ ]:
# -- Check current state of product names --------------------------------------

# Look for ALLCAPS, all lowercase, or names with extra whitespace
# We can detect these with pandas string methods

products["name_upper"] = products["product_name"] == products["product_name"].str.upper()
products["name_lower"] = products["product_name"] == products["product_name"].str.lower()
products["name_spaces"] = products["product_name"] != products["product_name"].str.strip()

n_allcaps  = products["name_upper"].sum()
n_alllower = products["name_lower"].sum()
n_spaces   = products["name_spaces"].sum()

print(f"Product names with formatting issues:")
print(f"  ALL CAPS:            {n_allcaps}")
print(f"  all lowercase:       {n_alllower}")
print(f"  Extra whitespace:    {n_spaces}")
print()

# Show some examples
if n_allcaps + n_alllower + n_spaces > 0:
    dirty_names = products[
        products["name_upper"] | products["name_lower"] | products["name_spaces"]
    ]["product_name"].head(8).tolist()
    print("Examples of dirty product names:")
    for n in dirty_names:
        print(f"  '{n}'")


In [ ]:
# -- Fix: strip whitespace then apply title case -------------------------------
#
# str.strip() removes leading/trailing spaces
# str.title() capitalizes the first letter of each word
#
# Note: str.title() has a known quirk — it will lowercase letters after
# apostrophes (e.g., "Men'S Shirt" instead of "Men's Shirt").
# For this dataset that's acceptable. In production, a custom title function
# would handle edge cases.

products["product_name"] = products["product_name"].str.strip().str.title()

# Drop the diagnostic helper columns
products.drop(columns=["name_upper","name_lower","name_spaces"], inplace=True)

# Verify
remaining = (
    (products["product_name"] == products["product_name"].str.upper()) |
    (products["product_name"] == products["product_name"].str.lower()) |
    (products["product_name"] != products["product_name"].str.strip())
).sum()

print(f"Remaining formatting issues in product_name: {remaining}")
print("✓  Product names standardized." if remaining == 0 else "⚠️  Some issues remain.")


---
### 2.9 — Data Type Casting

We loaded everything as strings. Now that we've cleaned the values, we cast each
column to its correct type. This matters because:

- You can't do `SUM(revenue_idr)` if it's stored as a string
- Date comparisons only work correctly when columns are datetime type
- Integer IDs take up less memory as int than as string
- Boolean columns used in filters need to be True/False, not "True"/"False"

> **Why cast AFTER cleaning, not before?**  
> If we'd cast `quantity` to int before removing negatives, the negatives would
> have silently become valid integers. If we'd cast `transaction_date` to datetime
> before fixing mixed formats, the non-ISO dates would have become NaT (null).
> Cleaning first, then casting, ensures our data is semantically correct before
> we enforce type constraints.


In [ ]:
# -- Cast transactions --------------------------------------------------------
txn["transaction_date"]  = pd.to_datetime(txn["transaction_date"], errors="coerce")
txn["quantity"]          = pd.to_numeric(txn["quantity"],         errors="coerce").astype("Int64")
txn["base_price_idr"]    = pd.to_numeric(txn["base_price_idr"],   errors="coerce").astype("Int64")
txn["discount_pct"]      = pd.to_numeric(txn["discount_pct"],     errors="coerce")
txn["final_price_idr"]   = pd.to_numeric(txn["final_price_idr"],  errors="coerce").astype("Int64")
txn["revenue_idr"]       = pd.to_numeric(txn["revenue_idr"],      errors="coerce").astype("Int64")
txn["cogs_idr"]          = pd.to_numeric(txn["cogs_idr"],         errors="coerce").astype("Int64")
txn["gross_profit_idr"]  = pd.to_numeric(txn["gross_profit_idr"], errors="coerce").astype("Int64")
txn["is_promo"]          = txn["is_promo"].map({"True": True, "False": False})

# -- Cast products ------------------------------------------------------------
products["cost_price_idr"]    = pd.to_numeric(products["cost_price_idr"],    errors="coerce").astype("Int64")
products["selling_price_idr"] = pd.to_numeric(products["selling_price_idr"], errors="coerce").astype("Int64")
products["weight_gram"]       = pd.to_numeric(products["weight_gram"],       errors="coerce").astype("Int64")
products["launch_date"]       = pd.to_datetime(products["launch_date"],      errors="coerce")
products["is_active"]         = products["is_active"].map({"True": True, "False": False})

# -- Cast inventory -----------------------------------------------------------
for col in ["opening_stock","incoming_qty","sold_qty","closing_stock",
            "days_of_inventory","lead_time_days","is_stockout"]:
    inventory[col] = pd.to_numeric(inventory[col], errors="coerce").astype("Int64")
inventory["sell_through_rate"] = pd.to_numeric(inventory["sell_through_rate"], errors="coerce")

# -- Cast traffic -------------------------------------------------------------
traffic["session_date"]   = pd.to_datetime(traffic["session_date"],   errors="coerce")
for col in ["sessions","orders","avg_order_value","is_promo_period"]:
    traffic[col] = pd.to_numeric(traffic[col], errors="coerce").astype("Int64")
traffic["conversion_rate"] = pd.to_numeric(traffic["conversion_rate"], errors="coerce")
traffic["revenue_idr"]     = pd.to_numeric(traffic["revenue_idr"],     errors="coerce").astype("Int64")

# -- Cast marketing -----------------------------------------------------------
marketing["week_start"] = pd.to_datetime(marketing["week_start"], errors="coerce")
for col in ["spend_idr","impressions","clicks","orders_attributed","revenue_attributed_idr"]:
    marketing[col] = pd.to_numeric(marketing[col], errors="coerce").astype("Int64")
marketing["ctr"]  = pd.to_numeric(marketing["ctr"],  errors="coerce")
marketing["roas"] = pd.to_numeric(marketing["roas"], errors="coerce")

print("Type casting complete.")
print()
print("transactions dtypes:")
print(txn.dtypes.to_string())


---
## Section 3 — Cross-Table Referential Integrity

Each table doesn't exist in isolation — they're related to each other:
- Every `sku_id` in `transactions` should exist in `products`
- Every `sku_id` in `inventory_snapshots` should exist in `products`
- Every `store_id` in `transactions` should be a valid store

This is called **referential integrity**: the "child" table (transactions) should not
reference values in the "parent" table (products) that don't exist.

> **Analogy:** Imagine a library's borrowing record table that references
> book IDs. If a borrowing record says "Book #9999 was borrowed" but Book #9999
> doesn't exist in the books table — that's a referential integrity violation.
> It means either the borrowing record is wrong, or the book was deleted but its
> history wasn't cleaned up.


In [ ]:
# -- Check 1: transaction SKUs vs product catalog ------------------------------

valid_skus    = set(products["sku_id"].dropna())
txn_skus      = set(txn["sku_id"].dropna())

orphan_skus_txn = txn_skus - valid_skus  # SKUs in transactions but NOT in products

print("Referential integrity: transactions.sku_id → products.sku_id")
print(f"  Unique SKUs in transactions:   {len(txn_skus)}")
print(f"  Unique SKUs in products:       {len(valid_skus)}")
print(f"  Orphan SKUs (not in products): {len(orphan_skus_txn)}")

if len(orphan_skus_txn) > 0:
    print(f"  ⚠️  Orphan SKUs found: {orphan_skus_txn}")
else:
    print("  ✓  All transaction SKUs exist in the product catalog.")


In [ ]:
# -- Check 2: inventory SKUs vs product catalog -------------------------------

inv_skus         = set(inventory["sku_id"].dropna())
orphan_skus_inv  = inv_skus - valid_skus

print("Referential integrity: inventory_snapshots.sku_id → products.sku_id")
print(f"  Unique SKUs in inventory:      {len(inv_skus)}")
print(f"  Orphan SKUs (not in products): {len(orphan_skus_inv)}")

if len(orphan_skus_inv) > 0:
    print(f"  ⚠️  Orphan SKUs: {list(orphan_skus_inv)[:10]}")
else:
    print("  ✓  All inventory SKUs exist in the product catalog.")

print()

# Note: It's expected that not all products appear in every table.
# Some products may have never been sold (no transaction) or never tracked in
# inventory (might be online-only or out of scope). 
# What matters is the DIRECTION: child → parent (not parent → child).
total_active = products[products["is_active"] == True]["sku_id"].nunique()
print(f"  Active SKUs in catalog: {total_active}")
print(f"  SKUs with at least 1 transaction: {len(txn_skus)}")
print(f"  ({len(valid_skus) - len(txn_skus)} active products with zero recorded sales — possible display-only/pre-launch)")


---
## Section 4 — Statistical Sanity Checks

Even after fixing known data quality issues, we do a final round of
**statistical sanity checks**. These check whether the data *makes logical sense*:
- Rates and percentages should be between 0 and 1 (or 0% and 100%)
- Final price should be ≤ base price (unless there's a legitimate reason)
- Sell-through rate should be ≥ 0 and ≤ 1
- ROAS should be positive

These checks catch problems that don't look like obvious errors individually
but violate business logic.


In [ ]:
# -- Check 1: discount_pct range (should be 0.0 to 1.0) -------------------------
disc = txn["discount_pct"].dropna()
print(f"discount_pct range: min={disc.min():.4f}, max={disc.max():.4f}")
print(f"  Values < 0 (impossible):  {(disc < 0).sum()}")
print(f"  Values > 1 (>100% disc.): {(disc > 1).sum()}")
if (disc < 0).sum() + (disc > 1).sum() == 0:
    print("  ✓  All discount values are in valid range [0, 1]")


In [ ]:
# -- Check 2: Price logic — final_price ≤ base_price --------------------------
# When discount > 0, final_price should be less than base_price.
# If final > base, that's a pricing data error.

txn_priced = txn.dropna(subset=["base_price_idr","final_price_idr"])
price_violated = txn_priced[txn_priced["final_price_idr"] > txn_priced["base_price_idr"]]

print(f"Rows where final_price > base_price: {len(price_violated)}")
if len(price_violated) > 0:
    print("\nSample violations:")
    print(price_violated[["sku_id","base_price_idr","discount_pct","final_price_idr"]].head())
else:
    print("✓  Price logic is consistent (final ≤ base for all rows).")


In [ ]:
# -- Check 3: Conversion rate range (should be 0 to 1) ------------------------
cvr = traffic["conversion_rate"].dropna()
print(f"conversion_rate range: min={cvr.min():.4f}, max={cvr.max():.4f}")
print(f"  Values < 0: {(cvr < 0).sum()}")
print(f"  Values > 1: {(cvr > 1).sum()}")
if (cvr < 0).sum() + (cvr > 1).sum() == 0:
    print("  ✓  All CVR values in valid range [0, 1]")


In [ ]:
# -- Check 4: STR range in inventory ------------------------------------------
# sell_through_rate should be between 0.0 and 1.0
# An STR > 1 would mean we sold more than we had available — physically impossible.

str_vals = inventory["sell_through_rate"].dropna()
print(f"sell_through_rate range: min={str_vals.min():.4f}, max={str_vals.max():.4f}")
print(f"  Values < 0:  {(str_vals < 0).sum()}")
print(f"  Values > 1:  {(str_vals > 1).sum()}")

# Distribution summary
print()
print("STR distribution by subcategory (median):")
inv_numeric = inventory.copy()
inv_numeric["sell_through_rate"] = pd.to_numeric(inv_numeric["sell_through_rate"], errors="coerce")
print(
    inv_numeric.groupby("subcategory")["sell_through_rate"]
    .median()
    .sort_values()
    .apply(lambda x: f"{x:.1%}")
    .to_string()
)
print()
print("⬆️  Note: Very low STR for any subcategory signals potential deadstock.")
print("    This will be a key analysis target in the EDA notebook.")


In [ ]:
# -- Check 5: ROAS sanity -----------------------------------------------------
# ROAS should be positive (revenue > 0, spend > 0)
# ROAS < 1 = ad spend exceeds attributed revenue (losing money on ads)
# ROAS > 20 = suspiciously high, may be attribution error

roas = marketing["roas"].dropna()
print(f"ROAS range: min={roas.min():.2f}, max={roas.max():.2f}, mean={roas.mean():.2f}")
print(f"  Negative ROAS (impossible): {(roas < 0).sum()}")
print(f"  ROAS < 1 (losing on ads):   {(roas < 1).sum()}")
print(f"  ROAS > 15 (suspicious):     {(roas > 15).sum()}")


---
## Section 5 — Final Summary & Save Clean Data

Before saving, we do one last verification: all five tables should have
sensible row counts, no unexpected nulls in critical columns, and correct types.


In [ ]:
# -- Pre-save summary ---------------------------------------------------------
print("=" * 65)
print("FINAL DATASET SUMMARY — AFTER CLEANING")
print("=" * 65)

summary = {
    "products":            products,
    "transactions":        txn,
    "inventory_snapshots": inventory,
    "traffic_sessions":    traffic,
    "marketing_spend":     marketing,
}

for name, df in summary.items():
    n_rows = len(df)
    n_nulls = df.isnull().sum().sum()
    print(f"\n  {name}")
    print(f"    Rows:          {n_rows:,}")
    print(f"    Total nulls:   {n_nulls:,}")
    print(f"    Columns:       {df.shape[1]}")

print()
print(f"Total rows across all tables: {sum(len(df) for df in summary.values()):,}")


In [ ]:
# -- Save clean tables ---------------------------------------------------------

CLEAN_DIR = "../data/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

FILE_MAP = {
    "products":            "clean_products.csv",
    "transactions":        "clean_transactions.csv",
    "inventory_snapshots": "clean_inventory_snapshots.csv",
    "traffic_sessions":    "clean_traffic_sessions.csv",
    "marketing_spend":     "clean_marketing_spend.csv",
}

print("Saving clean files...")
for key, fname in FILE_MAP.items():
    df   = summary[key]
    path = os.path.join(CLEAN_DIR, fname)
    df.to_csv(path, index=False)
    kb   = os.path.getsize(path) / 1024
    print(f"  ✓  {fname:<45} ({len(df):>6,} rows, {kb:>7.1f} KB)")

print("\nAll clean files saved to data/clean/")


---
## Summary — What We Found and Fixed

| Issue | Rows affected | Fix applied | Notes |
|---|---|---|---|
| Duplicate transactions | ~60 rows | `drop_duplicates(keep='first')` | System sync glitch |
| Unexpected NULL store_id (offline) | ~15 rows | Flagged, kept | Cannot recover — noted for downstream |
| Channel name variants | ~210 rows | Mapping dictionary | Normalized to 4 canonical values |
| Prices in USD | 30 rows | Dropped | 0.05% loss; error is too large to convert safely |
| Negative quantities | 25 rows | Dropped | Mis-recorded returns; not sales data |
| Non-ISO date formats | ~5% of dates | `parse_date()` multi-format parser | All parsed successfully |
| Province name variants | ~180 rows | Mapping dictionary | Normalized to official Indonesian names |
| Product name formatting | ~32 rows | `str.strip().str.title()` | |

---

## What's Next

The clean dataset is ready for analysis. In `02_analysis.ipynb` we will explore:

1. **Channel P&L** — which channel is actually most profitable?
2. **Inventory health** — are there deadstock SKUs? Which are at risk of stockout?
3. **RFM Segmentation** — who are Meridian's best customers? Who is churning?
4. **Cohort Analysis** — are customers coming back after first purchase?
5. **A/B Testing** — did the March 2024 promo actually work? On which channel?
6. **Geographic Analysis** — where is growth happening?

Each of these analyses is covered in detail in `02_analysis.ipynb`.
